<a href="https://colab.research.google.com/github/vicente-gonzalez-ruiz/medical_imaging/blob/main/notebooks/unet_n2n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Noise2Noise denoising of gray-scale images

In [ ]:
import urllib.request
from PIL import Image
import io
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load lena

url = "https://www.hpca.ual.es/~vruiz/images/lena_512x512_gray.png"
with urllib.request.urlopen(url) as response:
    img_data = response.read()
    clean = Image.open(io.BytesIO(img_data)).convert('L')
clean = np.array(lena_clean, dtype=np.uint8)

In [ ]:
# Add noise

mean = 0
sigma = 25
gaussian_noise = np.random.normal(mean, sigma, lena_clean.shape)
noisy = np.clip(clean + gaussian_noise, 0, 255).astype(np.uint8)

In [ ]:
# Show

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(clean, cmap="gray")
axes[0].set_title("Original Image")
axes[0].axis("off")
axes[1].imshow(noisy, cmap="gray")
axes[1].set_title(fr"Noisy Image (Gaussian $\sigma={sigma}$)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Define the U-Net (see https://arxiv.org/pdf/1803.04189) (see https://github.com/sashrika15/noise2noise/blob/master/unet.py)

class Unet(nn.Module):
    '''
    Unet architecture for n2n.
    No batch norm, dropout
    '''

    def __init__(self, in_channels=3, out_channels=3):
            """Initializes U-Net."""

            super(Unet, self).__init__()

            self._block1 = nn.Sequential(
                nn.Conv2d(in_channels, 48, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(48, 48, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2))

            self._block2 = nn.Sequential(
                nn.Conv2d(48, 48, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2))

            self._block3 = nn.Sequential(
                nn.Conv2d(48, 48, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.ConvTranspose2d(48, 48, 3, stride=2, padding=1, output_padding=1))

            self._block4 = nn.Sequential(
                nn.Conv2d(96, 96, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(96, 96, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.ConvTranspose2d(96, 96, 3, stride=2, padding=1, output_padding=1))

            self._block5 = nn.Sequential(
                nn.Conv2d(144, 96, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(96, 96, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.ConvTranspose2d(96, 96, 3, stride=2, padding=1, output_padding=1))

            self._block6 = nn.Sequential(
                nn.Conv2d(96 + in_channels, 64, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(64, 32, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(32, out_channels, 3, stride=1, padding=1),
                nn.LeakyReLU(0.1))


    def forward(self, x):

        #Encoder
        #print("X size = ", str(x.size()))
        pool1 = self._block1(x)
        #print(pool1.size())
        pool2 = self._block2(pool1)
        #print(pool2.size())
        pool3 = self._block2(pool2)
        #print(pool3.size())
        pool4 = self._block2(pool3)
        #print(pool4.size())
        pool5 = self._block2(pool4)
        #print(pool5.size())

        #Decoder
        upsample5 = self._block3(pool5)
        #print(upsample5.size())
        concat5 = torch.cat((upsample5, pool4), dim=1)
        #print(concat5.size())
        upsample4 = self._block4(concat5)
        #print(upsample4.size())
        concat4 = torch.cat((upsample4, pool3), dim=1)
        #print(concat4.size())
        upsample3 = self._block5(concat4)
        #print(upsample3.size())
        concat3 = torch.cat((upsample3, pool2), dim=1)
        #print(concat3.size())
        upsample2 = self._block5(concat3)
        #print(upsample2.size())
        concat2 = torch.cat((upsample2, pool1), dim=1)
        #print(concat2.size())
        upsample1 = self._block5(concat2)
        #print(upsample1.size())
        concat1 = torch.cat((upsample1, x), dim=1)
        #print(concat1.size())
        output = self._block6(concat1)
        #print(output.size())
        return output

    def summary(self):
        print('Unet summary: ')
        print(self._block1)
        print(self._block2)
        print(self._block3)
        print(self._block4)
        print(self._block5)
        print(self._block6)

Unet(in_channels=1, out_channels=1).summary()

In [ ]:
# Determine the "device"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
# Configure the denoiser's training

denoiser = Unet(in_channels=1, out_channels=1).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(denoiser.parameters(), lr=0.001)
epochs = 10

In [ ]:
# Get tensors from arrays

clean = torch.from_numpy(clean)
noisy = torch.from_numpy(noisy)

In [ ]:
# Setup fixed noisy input for visual consistency during tracking???

fixed_noisy_input = noisy.to(device)

In [ ]:
for epoch in range(1, epochs + 1):
    denoiser.train()
    epoch_loss = 0.0
    for noisy_source, noisy_target in lena_loader:
        noisy_source = noisy_source.to(device)
        noisy_target = noisy_target.to(device)

        optimizer.zero_grad()
        output = model_lena(noisy_source)
        loss = criterion(output, noisy_target)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    losses.append(epoch_loss)

    # Generate current reconstruction
    model_lena.eval()
    with torch.no_grad():
        current_output = model_lena(fixed_noisy_input).cpu()
    current_img = torch.clamp(current_output.squeeze(), 0.0, 1.0).numpy()

    # Display each reconstruction sequentially with details in the title
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(current_img, cmap='gray')
    ax.set_title(f'Epoch {epoch} (Loss: {epoch_loss:.4f})')
    ax.axis('off')
    plt.show()